# ESM3 alpha dial — geometry

How the refined receptor cloud moves between **structure** (the frozen ESM
embedding it started from) and **function** (the measured response profile) as the
dial turns, on the ESM3 sweep.

**This notebook draws and does not aggregate.** Every number comes from
`scripts/analysis/alpha_grid.py`. That matters for one reason above all: it averages
the **model seeds inside each fold first**, and only then takes a Student-t interval
over the **folds**. A fold changes which rows are held out — that is what a claim
about generalisation is over. A seed changes only the draw the same model made on the
same rows. Treating the 25 cells of a five-seed grid as 25 observations would use
t(24) where the honest value is t(4) and divide by √25 where the effective sample is
5, which halves every interval on the page.

Read with: `notebooks/graph/alpha_gate/` (the v8 gate, the original of this idiom) and
`notebooks/graph/node_dial/` (the v9 dial and the honest choice of alpha).

In [ ]:
import pathlib
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebooks" / "article_figures"))

import figkit as fk
from scripts.analysis import alpha_grid as ag

fk.use_house_style()
print("repo:", ROOT)

## Knobs

In [ ]:
# ================================== KNOBS ====================================
ROOT_DIR   = "results/graph/v13_esm3"   # the ESM3 sweep
NODES      = "nodedial"                 # "nodedial" (v9) | "onehot" (v8 gate) | None
MOL_SOURCE = "chemberta"
DATASET    = None          # None -> every dataset on disk; or ["cc", "hc"] / "m2or"
REGIME     = None          # None -> both; or "transductive" / "inductive"
SEED       = None          # model seed; None -> every one present
COMBO      = "cls+mol"     # which head the graph rows come from

GEOM_SCALE = "value"       # "value" = the measure itself | "z" = vs its own null
CI_LEVEL   = 0.95          # Student-t over FOLDS (seeds are averaged inside each fold)
RESOLUTION_METRIC = "rsa_fun"   # what the I-mark's floor is computed on

SAVE_FIGS  = False
FIG_DIR    = "results/article_figures/esm3"
# =============================================================================

df = ag.load(root=ROOT_DIR, mol_source=MOL_SOURCE, nodes=NODES,
             dataset=DATASET, regime=REGIME, seed=SEED, combo=COMBO)

SERIES = sorted(df.series.unique())
ALPHAS = sorted(df.loc[df.arm == "gate", "alpha"].dropna().unique())

In [ ]:
# ---------------------------------------------------------------- what is on disk
# The two dials must never share an axis. On a v8 GATE run alpha mixes the graph's
# OUTPUT against a frozen ESM branch; on a v9 NODE dial it mixes the graph's INPUT,
# and it runs the other way (alpha=0 is receptor identity, alpha=1 the legacy graph).
# A frame holding both puts opposite meanings on one x axis, so refuse it here rather
# than discover it in a plot.
dials = sorted(df["nodes"].astype(str).unique())
if len(dials) != 1:
    raise SystemExit(
        f"this frame holds {dials} -- two dials cannot share an axis. "
        f"Set NODES to one of them.")

cov = ag.coverage(df)
print(f"{len(SERIES)} series | {len(ALPHAS)} alphas | dial: {dials[0]}")
print("alphas:", ", ".join(f"{a:g}" for a in ALPHAS))
pin = ", ".join(f"{k}={v}" for k, v in df.attrs["pinned"].items()) or "-"
print("pinned:", pin)
cov

In [ ]:
# ------------------------------------------------- how small a difference is visible
# The floor: the model noise LEFT IN A FOLD MEAN after averaging the seeds. Every curve
# on this page is made of fold means, so a wiggle below this number is not a finding.
#
# This is also why the curves carry a CI band over folds and an I-mark for the floor,
# and never whiskers over folds: the between-fold spread is the larger quantity by a
# wide margin, and drawing it on the curve answers "how different are the folds" on a
# figure that is asking "did moving the dial change anything".
ns = ag.noise_split(df, RESOLUTION_METRIC)
FLOOR = {}
if len(ns):
    g = ns.groupby("series")[["within", "between", "floor"]].mean()
    FLOOR = g["floor"].to_dict()
    display(g.round(4))
    print("\nratio between/floor -- how much of the spread is the folds, not the model:")
    display((g["between"] / g["floor"]).round(1).rename("x"))
else:
    print(f"{RESOLUTION_METRIC!r} not in this sweep; no floor computed")

---
## 1. Geometry along the dial

Three measures, each oriented so **larger means more aligned**: RSA (do the two spaces
order neighbours the same way), CCA (mean canonical correlation on the top-3 PCs) and
Procrustes (1 − disparity on the same PCs). Each is taken twice — against ESM and
against the response profile.

In [ ]:
# One row per (series, alpha, geom, ref). `ag.geometry` averages model seeds inside
# each fold and takes the t interval over folds -- see alpha_grid.fold_means.
geo = ag.geometry(df, scale=GEOM_SCALE, level=CI_LEVEL)
print(f"{len(geo)} aggregated points | geoms: {sorted(geo.geom.unique())} "
      f"| refs: {sorted(geo.ref.unique())}")
geo.head()

In [ ]:
# 1.1  ALIGNMENT AGAINST BOTH EXTREMES, ALONG THE DIAL.
# One row of panels per series, one column per geometry. Two curves in each: the
# receptor cloud against frozen ESM (structure) and against the response profile
# (function). Every geometry is oriented so LARGER = MORE ALIGNED, so the dial doing
# what it claims means the ESM curve falls and the function curve rises.
ncol = len(ag.GEOMS)
fig, ax = fk.panels(len(SERIES) * ncol, ncols=ncol, w=3.0, h=2.25, sharex=True)

for i, s in enumerate(SERIES):
    for j, g in enumerate(ag.GEOMS):
        a, drawn = ax[i * ncol + j], 0
        for ref in ag.REFS:
            q = geo[(geo.series == s) & (geo.geom == g) & (geo.ref == ref)]
            drawn += fk.curve_with_band(a, q, fk.C[ref])
        fk.alpha_axis(a, ALPHAS)
        if not drawn:
            fk.note_empty(a, "no geometry yet")
        if i == 0:
            a.set_title(ag.GEOM_LABEL[g], pad=8)
        if j == 0:
            a.set_ylabel(s, fontsize=8.5)
        if i == len(SERIES) - 1:
            a.set_xlabel(r"$\alpha$")

unit = "alignment (higher = closer)" if GEOM_SCALE == "value" else "z vs permutation null"
fig.suptitle(f"Receptor cloud against the two extremes, along the dial - {unit}",
             y=1.005, fontsize=11)
fk.figlegend(fig, [
    Line2D([], [], color=fk.C["esm"], marker="o", label="vs ESM  (structure)"),
    Line2D([], [], color=fk.C["fun"], marker="o", label="vs response profile  (function)"),
    Line2D([], [], color=fk.MUTED, lw=6, alpha=0.2,
           label=f"band = {CI_LEVEL:.0%} CI over folds"),
], ncol=3)
fk.savefig(fig, f"geometry_curves_{GEOM_SCALE}", FIG_DIR, SAVE_FIGS)
plt.show()

### 1.2 The crossover

In [ ]:
# 1.2  WHERE THE TWO ALIGNMENTS CROSS. One number per (series, geometry): the alpha at
# which the cloud stops being closer to ESM than to the response profile. It is a
# reading of the two curves above, not a second measurement -- `ag.crossover` scales
# each pair within its own panel first, because RSA and CCA do not share a range.
cross = ag.crossover(geo)
display(cross.round(3))

fig, ax = fk.panels(1, ncols=1, w=5.6, h=0.55 * max(len(SERIES), 2) + 1.2)
a = ax[0]
MK = {"rsa": "o", "cca": "s", "procrustes": "^"}
ypos = {s: k for k, s in enumerate(SERIES)}
for _, r in cross.iterrows():
    if not np.isfinite(r.alpha_cross):
        continue
    a.plot(r.alpha_cross, ypos[r.series], marker=MK.get(r.geom, "o"),
           color=fk.C["fun"], ms=6, ls="none")
a.set_yticks(range(len(SERIES)))
a.set_yticklabels(SERIES, fontsize=8.5)
a.set_ylim(-0.6, len(SERIES) - 0.4)
fk.alpha_axis(a, ALPHAS)
a.set_xlabel(r"$\alpha$ at which function overtakes structure")
fk.figlegend(fig, [Line2D([], [], color=fk.C["fun"], marker=MK[g], ls="none",
                          label=ag.GEOM_LABEL[g].split(" - ")[0]) for g in ag.GEOMS],
             ncol=3)
fk.savefig(fig, "geometry_crossover", FIG_DIR, SAVE_FIGS)
plt.show()

### 1.3 How far the curves travel

In [ ]:
# 1.3  HOW FAR EACH CURVE ACTUALLY TRAVELS. The crossover says the curves met; this says
# whether either of them moved enough for that to mean anything. `ag.audit` reports the
# value at both ends of the dial and the span between them, per (series, geom, ref).
aud = ag.audit(geo)
display(aud.sort_values(["series", "geom", "ref"]).round(3))

---
## What this page cannot say

The geometry is measured against the **train** response profile, so on a sparse panel
it partly encodes which pairs were assayed rather than how the receptors behave — the
M2OR caution in `02a_protein_geometry.py` applies here too. `scale="z"` says whether an
alignment is distinguishable from chance, but its size grows with the receptor count
(1237 on M2OR against 24 on HC), so z is never an effect size across panels.